In [0]:
# connection

jdbc_url = (
    "jdbc:postgresql://aws-1-ap-southeast-2.pooler.supabase.com:5432/postgres"
    "?sslmode=require"
)

connection_props = {
    "user": "postgres.peoxnpyxfflcgycnvmvo",
    "password": "db_ingestion_pass",
    "driver": "org.postgresql.Driver"
}

# helper function

from pyspark.sql.functions import max as spark_max

def get_last_watermark(table_name, watermark_col):
    try:
        df = spark.table(table_name)
        return df.select(spark_max(watermark_col)).collect()[0][0]
    except:
        return None


def incremental_read(jdbc_url, table, watermark_col, last_value, props):
    if last_value is None:
        query = f"(SELECT * FROM {table}) as src"
    else:
        query = f"""
        (SELECT *
         FROM {table}
         WHERE {watermark_col} > '{last_value}') as src
        """

    return spark.read.jdbc(
        url=jdbc_url,
        table=query,
        properties=props
    )

# customers incremental

watermark_col = "updated_at"

last_value = get_last_watermark("bronze.customers", watermark_col)

customers_df = incremental_read(
    jdbc_url,
    "public.customers",
    watermark_col,
    last_value,
    connection_props
)

customers_df.write.format("delta") \
    .mode("append") \
    .saveAsTable("bronze.customers")


# policies incremental

last_value = get_last_watermark("bronze.policies", watermark_col)

policies_df = incremental_read(
    jdbc_url,
    "public.policies",
    watermark_col,
    last_value,
    connection_props
)

policies_df.write.format("delta") \
    .mode("append") \
    .saveAsTable("bronze.policies")

# claims incremental

last_value = get_last_watermark("bronze.claims", watermark_col)

claims_df = incremental_read(
    jdbc_url,
    "public.claims",
    watermark_col,
    last_value,
    connection_props
)

claims_df.write.format("delta") \
    .mode("append") \
    .saveAsTable("bronze.claims")

